In [1]:
import pandas as pd
import numpy as np
from pyproj import Transformer

## Sliding filter for CTD data

In [2]:
ctd13_raw = pd.read_csv('at5036013_ctd_2025Apr30_sonardyne_export.csv',skiprows=2)
ctd13_usbl = ctd13_raw.loc[ctd13_raw['SourceName'] == 'SSSG2 2113'].copy().reset_index()

In [3]:
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32614", always_xy=True)

easting, northing = transformer.transform(ctd13_usbl['Longitude'].values,ctd13_usbl['Latitude'].values)

ctd13_usbl['Easting']  = easting
ctd13_usbl['Northing'] = northing

In [5]:
window_width = 15
sigma = 3
min_obs = 3

for i in ['Easting','Northing']:
    slide = ctd13_usbl[i].rolling(window = window_width,min_periods=min_obs,center=True)
    slide_mean = slide.mean()
    slide_stdv = slide.std()
    ctd13_usbl[f"{i}_outlier"] = (ctd13_usbl[i] < slide_mean - sigma * slide_stdv) | (ctd13_usbl[i] > slide_mean + sigma * slide_stdv)

rejected_mask = ctd13_usbl["Northing_outlier"] | ctd13_usbl["Easting_outlier"]
ctd13_usbl_clean = ctd13_usbl[~rejected_mask].copy().reset_index()

## Interpolating onto CTD files

In [7]:
ctd13_proc = pd.read_csv('/Users/Sam/Documents/Postdoc/Cruise/Processed CTD Data/at5036_013_nav.csv')

In [8]:
ctd13_usbl_clean['datetime'] = pd.to_datetime(ctd13_usbl_clean['TimeStamp'])
ctd13_proc['datetime'] = pd.to_datetime(ctd13_proc['date'])

usbl_t = ctd13_usbl_clean['datetime'].astype(np.int64).values
ctd_t = ctd13_proc['datetime'].astype(np.int64).values

In [9]:
interp_cols = ['Latitude', 'Longitude', 'Easting', 'Northing']

for i in interp_cols:
    ctd13_proc[i] = np.interp(ctd_t,usbl_t,ctd13_usbl_clean[i].values)

In [10]:
ctd13_proc.to_csv('/Users/Sam/Documents/Postdoc/Cruise/Processed CTD Data/at5036_013_nav_sdk.csv',index=False)